# KAN — Kolmogorov-Arnold Network (efficient-kan) — Telco Churn

KAN substitui pesos fixos por funções spline aprendíveis nas arestas.
Usa `efficient-kan` (reimplementação mais rápida do paper original).

Optuna busca arquitetura, `grid_size` (nós da spline) e hiperparâmetros de treino.

In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn import metrics as skm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from model_utils import load_data, compute_metrics, save_results, print_scores

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 0. Instalar efficient-kan

In [3]:
!pip install git+https://github.com/Blealtan/efficient-kan.git -q
from efficient_kan import KAN
print('efficient-kan OK')


efficient-kan OK


## 1. Carregar dados

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
INPUT_SIZE = X_train.shape[1]

def to_tensors(X, y):
    return (torch.FloatTensor(X.values).to(device),
            torch.LongTensor(y.values.astype(int)).to(device))

Xtr, ytr = to_tensors(X_train, y_train)
Xvl, yvl = to_tensors(X_val, y_val)
Xte, yte = to_tensors(X_test, y_test)
y_val_np  = y_val.values.astype(int)
y_test_np = y_test.values.astype(int)

Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)


## 2. Definição do modelo KAN

In [5]:
def build_kan(hidden_size, n_hidden_layers, grid_size):
    """Constrói KAN com arquitetura [INPUT_SIZE, h, ..., h, 2]."""
    layers = [INPUT_SIZE] + [hidden_size] * n_hidden_layers + [2]
    return KAN(layers_hidden=layers, grid_size=grid_size).to(device)

def eval_ks(model, X, y_np):
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(X), dim=1)[:, 1].cpu().numpy()
    fpr, tpr, _ = skm.roc_curve(y_np, probs)
    return float(np.max(tpr - fpr))

def make_loader(Xt, yt, batch_size, shuffle=True):
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

## 3. Optuna HPO (20 trials, maximiza KS no val)

In [6]:
def objective(trial):
    hidden_size     = trial.suggest_categorical('hidden_size', [16, 32, 64])
    n_hidden_layers = trial.suggest_int('n_hidden_layers', 1, 2)
    grid_size       = trial.suggest_categorical('grid_size', [5, 10, 20])
    lr              = trial.suggest_float('lr', 1e-3, 1e-2, log=True)
    batch_size      = trial.suggest_categorical('batch_size', [32, 64, 128])

    model     = build_kan(hidden_size, n_hidden_layers, grid_size)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader    = make_loader(Xtr, ytr, batch_size)

    best_ks, patience_counter = 0.0, 0
    for epoch in range(30):
        model.train()
        for Xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(Xb), yb).backward()
            optimizer.step()
        ks = eval_ks(model, Xvl, y_val_np)
        if ks > best_ks:
            best_ks = ks; patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 5: break
    return best_ks

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study.optimize(objective, n_trials=20, show_progress_bar=True)
print(f'Best KS (val): {study.best_value:.4f}')
print('Best params:', study.best_params)

  0%|          | 0/20 [00:00<?, ?it/s]

Best KS (val): 0.5431
Best params: {'hidden_size': 32, 'n_hidden_layers': 1, 'grid_size': 10, 'lr': 0.0010837840501788513, 'batch_size': 128}


## 4. Retreinar com melhores parâmetros

In [7]:
bp = study.best_params
best_model = build_kan(bp['hidden_size'], bp['n_hidden_layers'], bp['grid_size'])
optimizer  = optim.Adam(best_model.parameters(), lr=bp['lr'])
criterion  = nn.CrossEntropyLoss()
loader     = make_loader(Xtr, ytr, bp['batch_size'])

best_ks, best_state, patience_counter = 0.0, None, 0
for epoch in range(50):
    best_model.train()
    epoch_loss = 0.0
    for Xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(best_model(Xb), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    ks = eval_ks(best_model, Xvl, y_val_np)
    if ks > best_ks:
        best_ks = ks
        best_state = {k: v.clone() for k, v in best_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print(f'Early stop @ epoch {epoch+1}'); break
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}  loss={epoch_loss/len(loader):.4f}  val_KS={ks:.4f}')

best_model.load_state_dict(best_state)
print(f'Best val KS: {best_ks:.4f}')

Epoch  10  loss=0.4471  val_KS=0.5183
Early stop @ epoch 19
Best val KS: 0.5371


## 5. Avaliar no teste + salvar artefatos

In [8]:
best_model.eval()
with torch.no_grad():
    logits = best_model(Xte)
    probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    preds  = logits.argmax(dim=1).cpu().numpy()

scores = compute_metrics(y_test_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/kan', exist_ok=True)
torch.save({'state_dict': best_state, 'best_params': bp,
            'input_size': INPUT_SIZE, 'scores': scores},
           'results/kan/model.pt')

save_results('kan', y_test_np, preds, probs, scores)

Scores no teste:
  accuracy               0.7493
  balanced_accuracy      0.7689
  precision              0.5171
  recall                 0.8107
  f1                     0.6314
  auroc                  0.8516
  ks                     0.5436
Saved → results/kan
